In [2]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['TiendaBigData'] 
coleccion = db['Aliexpress'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [3]:
# --- PASO 0: LIMPIEZA Y PREPARACIÓN ---
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print(" Limpieza de procesos y temporales completada.")

# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "Tu_Nombre_de_Grupo"
# Cambiamos a la URL de AliExpress (Búsqueda de Laptops)
URL_OBJETIVO = "https://es.aliexpress.com/w/wholesale-laptops.html"

datos_finales = []
driver = None

# --- PASO 1: CONFIGURACIÓN DEL NAVEGADOR ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("--display=:0") # Vital para ver el VNC

options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

try:
    driver = webdriver.Chrome(options=options)
    print(" Navegador iniciado correctamente.")

    # --- PASO 2: ABRIR LA PÁGINA ---
    driver.get(URL_OBJETIVO)
    time.sleep(5)

    # AliExpress a veces muestra un pop-up de bienvenida o cookies. 
    # Damos tiempo para que el usuario lo cierre en el VNC si aparece.
    print("🖥 Revisa el VNC (Puerto 6080). Cierra pop-ups o resuelve captchas.")
    input("Cuando la lista de productos sea visible, presiona ENTER para extraer...")

    # --- PASO 3: ESPERAR RESULTADOS ---
    # Selector de AliExpress para las tarjetas de producto
    selector_bloque = "div.list--gallery--3J1_Ssy, div.search-item-card" 
    
    try:
        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div[class*='list--gallery'], div[class*='search-item-card']"))
        )
    except Exception as e:
        print(" No se detectaron los bloques. Intentaremos extraer lo que haya cargado.")

    # --- PASO 4: EXTRAER BLOQUES ---
    # Usamos un selector parcial para las cajas de productos de AliExpress
    bloques = driver.find_elements(By.CSS_SELECTOR, "div[class*='list--item'], div[class*='search-item-card']")

    print(f" Bloques encontrados: {len(bloques)}")

    for bloque in bloques:
        try:
            # Selectores de AliExpress (pueden variar según la región, estos son los comunes)
            nombre = bloque.find_element(By.CSS_SELECTOR, "h3, div[class*='--title--']").text.strip()
            
            # El precio en AliExpress suele estar dividido en partes o en un div de clase 'price--main'
            precio_texto = bloque.find_element(By.CSS_SELECTOR, "div[class*='--price--'], span[class*='price--main']").text.strip()

            if nombre and precio_texto:
                datos_finales.append({
                    "identificador": nombre,
                    "valor": precio_texto,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO,
                    "tienda": "AliExpress"
                })
        except:
            continue

    print(f" Extracción terminada: {len(datos_finales)} productos capturados.")

except Exception as e:
    print(f" Error en Selenium: {e}")

finally:
    if driver is not None:
        driver.quit()

# --- PASO 5: GUARDAR EN MONGODB ---
try:
    client = MongoClient("database", 27017, serverSelectionTimeoutMS=5000)
    db = client["TiendaBigData"]
    # Cambiamos el nombre de la colección para diferenciar
    coleccion = db["AliExpressLaptops"]

    if datos_finales:
        for d in datos_finales:
            # Limpieza de precio para AliExpress (ej: "450,50 €" -> 450.50)
            v_sucio = str(d["valor"])
            # Quitamos el símbolo de Euro, espacios y arreglamos separadores
            v_limpio = v_sucio.replace("€", "").replace("US", "").replace("$", "").replace(".", "").replace(",", ".").strip()
            try:
                d["valor"] = float(v_limpio)
            except:
                d["valor"] = 0.0

        coleccion.insert_many(datos_finales)
        print(" Datos cargados en MongoDB (Colección AliExpress) correctamente.")
    else:
        print(" No hay datos para guardar.")

except Exception as e:
    print(f" Error en MongoDB: {e}")

🧹 Limpieza de procesos y temporales completada.
❌ Error en Selenium: Message: session not created: Chrome instance exited. Examine ChromeDriver verbose log to determine the cause.; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
#0 0x5a97a0880a6a <unknown>
#1 0x5a97a028fab5 <unknown>
#2 0x5a97a02cc66b <unknown>
#3 0x5a97a02c7339 <unknown>
#4 0x5a97a031753e <unknown>
#5 0x5a97a0316c2c <unknown>
#6 0x5a97a02d5cbf <unknown>
#7 0x5a97a02d6a81 <unknown>
#8 0x5a97a0846a64 <unknown>
#9 0x5a97a0849951 <unknown>
#10 0x5a97a083321e <unknown>
#11 0x5a97a084a51e <unknown>
#12 0x5a97a0819be0 <unknown>
#13 0x5a97a086d9b8 <unknown>
#14 0x5a97a086db88 <unknown>
#15 0x5a97a087f4de <unknown>
#16 0x72eacff14ac3 <unknown>

⚠️ No hay datos para guardar.


In [5]:
# --- PASO 0: LIMPIEZA Y PREPARACIÓN ---
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Cierra procesos anteriores que hayan quedado abiertos
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print(" Limpieza de procesos y temporales completada.")

# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "Tu_Nombre_de_Grupo"
URL_OBJETIVO = "https://www.aliexpress.com/wholesale?SearchText=laptop"

datos_finales = []
driver = None

# --- PASO 1: CONFIGURACIÓN DEL NAVEGADOR ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"

# Argumentos de estabilidad para Docker
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-software-rasterizer")
options.add_argument("--window-size=1920,1080")
options.add_argument("--remote-debugging-port=9222")

# User-Agent común
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

# --- SELECTORES CANDIDATOS ---
# AliExpress cambia su estructura con frecuencia, por eso probamos varias alternativas
SELECTORES_BLOQUE = [
    "div.search-item-card-wrapper-gallery",   # opción común en listados
    "div.search-item-card-wrapper",           # alternativa común
    "a.search-card-item",                     # otra variante posible
    "div[data-product-id]"                    # opción genérica
]

SELECTORES_NOMBRE = [
    "h3",
    "a[title]",
    ".multi--titleText--nXeOvyr",
    ".title"
]

SELECTORES_PRECIO = [
    ".multi--price-sale--U-S0jtj",
    ".multi--price-original--1zEQqOK",
    ".price-current",
    ".price"
]

def extraer_primer_texto(elemento, lista_selectores):
    """Prueba varios selectores y devuelve el primer texto no vacío."""
    for selector in lista_selectores:
        try:
            texto = elemento.find_element(By.CSS_SELECTOR, selector).text.strip()
            if texto:
                return texto
        except:
            continue
    return ""

try:
    # Inicia Chrome
    driver = webdriver.Chrome(options=options)
    print(" Navegador iniciado correctamente.")
    print(" Revisa el navegador en: http://localhost:6080/vnc.html")

    # --- PASO 2: ABRIR LA PÁGINA ---
    driver.get(URL_OBJETIVO)
    time.sleep(8)

    print("Título inicial:", driver.title)
    print("URL inicial:", driver.current_url)

    html = driver.page_source.lower()

    # Diagnóstico básico del estado de la página
    if any(texto in html for texto in ["captcha", "unusual activity", "verify", "are you a human"]):
        print(" Se detectó una validación o restricción del sitio.")
    else:
        print(" Sin bloqueo evidente en la carga inicial.")

    # Pausa para observación manual
    print(" Abre la interfaz visual en http://localhost:6080/vnc.html")
    input("Cuando veas la página lista para revisar resultados, presiona ENTER para continuar...")

    # Revisión después de la pausa
    print("Título después de la revisión:", driver.title)
    print("URL después de la revisión:", driver.current_url)

    html = driver.page_source.lower()
    if any(texto in html for texto in ["captcha", "unusual activity", "verify", "are you a human"]):
        print(" El sitio sigue mostrando una validación o restricción.")
        print(" El script no continuará con la extracción.")
    else:
        print(" Se intentará extraer resultados.")

        # --- PASO 3: ENCONTRAR EL SELECTOR QUE FUNCIONA ---
        bloques = []
        selector_usado = None

        for selector in SELECTORES_BLOQUE:
            try:
                WebDriverWait(driver, 10).until(
                    EC.presence_of_all_elements_located((By.CSS_SELECTOR, selector))
                )
                bloques = driver.find_elements(By.CSS_SELECTOR, selector)
                if len(bloques) > 0:
                    selector_usado = selector
                    break
            except:
                continue

        if not bloques:
            print(" No se encontraron bloques de producto con los selectores probados.")
            print("Primeros 1200 caracteres del HTML:")
            print(driver.page_source[:1200])
            raise Exception("No se detectaron tarjetas de producto en AliExpress.")

        print(f" Bloques encontrados: {len(bloques)}")
        print(f" Selector usado: {selector_usado}")

        # --- PASO 4: EXTRAER NOMBRE Y PRECIO ---
        for bloque in bloques:
            try:
                nombre = extraer_primer_texto(bloque, SELECTORES_NOMBRE)
                precio = extraer_primer_texto(bloque, SELECTORES_PRECIO)

                if nombre or precio:
                    datos_finales.append({
                        "identificador": nombre if nombre else "SIN_NOMBRE",
                        "valor": precio if precio else "SIN_PRECIO",
                        "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                        "grupo": NOMBRE_GRUPO,
                        "sitio": "AliExpress"
                    })
            except:
                continue

        print(f" Extracción terminada: {len(datos_finales)} productos capturados.")

except Exception as e:
    print(f" Error en Selenium: {e}")

finally:
    if driver is not None:
        try:
            driver.quit()
        except:
            pass

# --- PASO 5: GUARDAR EN MONGODB ---
try:
    client = MongoClient("database", 27017, serverSelectionTimeoutMS=5000)
    db = client["TiendaBigData"]
    coleccion = db["AliExpressLaptops"]

    if datos_finales:
        for d in datos_finales:
            # Limpieza básica del precio
            v_limpio = (
                str(d["valor"])
                .replace(".", "")
                .replace(",", "")
                .replace("€", "")
                .replace("US $", "")
                .replace("$", "")
                .strip()
            )
            d["valor"] = float(v_limpio) if v_limpio.isdigit() else 0.0

        coleccion.insert_many(datos_finales)
        print(" Datos cargados en MongoDB correctamente.")
    else:
        print(" No hay datos para guardar en MongoDB.")

except Exception as e:
    print(f" Error en MongoDB: {e}")

🧹 Limpieza de procesos y temporales completada.
✅ Navegador iniciado correctamente.
🖥️ Revisa el navegador en: http://localhost:6080/vnc.html
Título inicial: Captcha Interception
URL inicial: https://www.aliexpress.com//wholesale/_____tmd_____/punish?x5secdata=xfaOHQ1mp_jrAGx2lfPw00XLgltLZrGYqW5OJDe-AKzUC_EAfPCS5tlPbonj69zjowhiO3S40u78kcmh_dh3_exydz-l3p_XD_ux5D-Wmd7cMO7tUvOIRRbRba9E91-cxb77A77YtN9g2z59_QPGb-RmxwipO4Ch9ZGFhBhBLmYZNF2QaqDBDDsGlXmi5dHiUQWJ-ljWZXSo6ezDflkYeZDUSM72c-3u1WMwabfJ4yQ7wLs2FO5tYuXtCLEdUnrtE0XJbCaq9noDp_jSi4DyK1PfT_PJ99Wi36SJ2p8DirVea-v2B-oEPxbjN9-rjpEeet9OhQJ74kGn_c_YSHF-P_85lpKr7QJOUGMe9kJVgaJqEL4eVmPITclJe2vQ-XydYjB6JQ1mODBs0Q7vvuIUt1VNRi-AaCf2EX8zo4G-gEYBm3auGlNcfjlrHj3_fHPR7-kXW_69pTKPwEGr6frSB-RTVr6bjdgxg_gRlVV32zRfsXnMrhJf45CvInC0a2WEiwWS4JT4guSvlHqh7SbMMuCm7SuWbTPy417-OtVNBcrm3OLU2VvPYU86eLXDk8VyEAzLi5L9GOsRgUNbV-KSdJLZohX1LUbXZuBrT_4IeTLUnBIQ9sOSeDnnVidvlQFfOn0azGb0MF2VkA11Pz6TaedUPpcv_FHiQBxSJ9vGZ4Oee1Zhesi-maW_dzbhWeDJnj3L6mLpHPfqY4E3iEPnIq_OmuJiowS7Tdm

Cuando veas la página lista para revisar resultados, presiona ENTER para continuar... 


Título después de la revisión: Captcha Interception
URL después de la revisión: https://www.aliexpress.com//wholesale/_____tmd_____/punish?x5secdata=xfaOHQ1mp_jrAGx2lfPw00XLgltLZrGYqW5OJDe-AKzUC_EAfPCS5tlPbonj69zjowhiO3S40u78kcmh_dh3_exydz-l3p_XD_ux5D-Wmd7cMO7tUvOIRRbRba9E91-cxb77A77YtN9g2z59_QPGb-RmxwipO4Ch9ZGFhBhBLmYZNF2QaqDBDDsGlXmi5dHiUQWJ-ljWZXSo6ezDflkYeZDUSM72c-3u1WMwabfJ4yQ7wLs2FO5tYuXtCLEdUnrtE0XJbCaq9noDp_jSi4DyK1PfT_PJ99Wi36SJ2p8DirVea-v2B-oEPxbjN9-rjpEeet9OhQJ74kGn_c_YSHF-P_85lpKr7QJOUGMe9kJVgaJqEL4eVmPITclJe2vQ-XydYjB6JQ1mODBs0Q7vvuIUt1VNRi-AaCf2EX8zo4G-gEYBm3auGlNcfjlrHj3_fHPR7-kXW_69pTKPwEGr6frSB-RTVr6bjdgxg_gRlVV32zRfsXnMrhJf45CvInC0a2WEiwWS4JT4guSvlHqh7SbMMuCm7SuWbTPy417-OtVNBcrm3OLU2VvPYU86eLXDk8VyEAzLi5L9GOsRgUNbV-KSdJLZohX1LUbXZuBrT_4IeTLUnBIQ9sOSeDnnVidvlQFfOn0azGb0MF2VkA11Pz6TaedUPpcv_FHiQBxSJ9vGZ4Oee1Zhesi-maW_dzbhWeDJnj3L6mLpHPfqY4E3iEPnIq_OmuJiowS7TdmWLufGasPu3-xts-EC-Y7xaYazQs2FpsOd68d-Ydx1ogSSQMfo8veooUIgRUA3Vb7fI34bD_agFOVOd6wN-TKBsPLieof5B1VSN1punXZC4rBS1GU